# FastGS — training and submission pipeline

This notebook is **glue only**. Every step is a function in `pipeline/`:

| File | Responsibility |
|---|---|
| `pipeline/config.py` | every parameter, in one place |
| `pipeline/env.py` | GPU check, dependency install, RAM/VRAM tracking and cleanup |
| `pipeline/data.py` | download data, discover scenes, data profile |
| `pipeline/trainer.py` | FastGS training loop with live scoring |
| `pipeline/score.py` | LPIPS / SSIM / PSNR and `Score = 0.4(1−LPIPS) + 0.3·SSIM + 0.3·PSNR_norm` |
| `pipeline/submission.py` | render the test poses → `submission.zip` + format check |
| `pipeline/report.py` | comparison tables and plots |
| `pipeline/deliver.py` | package results and download them |
| `pipeline/run.py` | wires it together (`setup → load_data → smoke_test → run_all → analytics → finish`) |

How FastGS works: [DOCS/fastgs-acceleration-method.md](DOCS/fastgs-acceleration-method.md) — with a runnable,
CUDA-free micro-simulation in `python demos/fastgs_mechanisms.py`.

**Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.

## 0 — Get the code

The first install takes ~3–5 minutes to build the three CUDA submodules; later sessions skip it via the `/content/.deps_ok` flag.

In [ ]:
REPO_URL = "https://github.com/KietAnhCS/fastgs-lite.git"
REPO_DIR = "/content/fastgs-lite"

import os, sys

if os.path.isdir("pipeline"):
    REPO_DIR = os.getcwd()                      # already running inside the repo
elif not os.path.isdir(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)

## 1 — Check the GPU and mount Drive

Hộp thoại cấp quyền Drive là **thứ duy nhất chặn Run All**, nên nó được đặt ngay ở đây:
bấm cho phép một lần, mọi ô sau chạy liền mạch.

In [ ]:
from pipeline.env import check_gpu, install_dependencies
from pipeline.data import mount_drive

install_dependencies()        # pip packages + 3 CUDA submodules (skipped once installed)
mount_drive()                 # bấm cho phép 1 lần — dùng cho cả đọc data lẫn lưu model
check_gpu(require=True)       # GPU / VRAM / torch / CUDA / RAM

## 2 — Configuration

Sửa **duy nhất ở đây**. Dữ liệu cuộc thi nằm trên Google Drive
(`VAI_NVS_DATA_ROUND2`), mỗi scene có dạng:

```
HCM0539/
├── train/
│   ├── images/       240 anh
│   └── sparse/0/     cameras.bin, images.bin, points3D.bin
└── test/
    ├── images/       60 anh
    └── test_poses.csv
```

Có **ba** cách lấy dữ liệu, pipeline ưu tiên theo thứ tự:

| Cách | Đặt gì | Khi nào dùng |
|---|---|---|
| 1. Gắn Drive | `drive_mount=True` + `scene_root=...` | **Khuyến nghị** — không tải, không giới hạn số file |
| 2. gdown | `drive_folder_url=...` | Khi thư mục không nằm trong Drive của bạn |
| 3. ZIP | `dataset_url=...` | Dữ liệu đóng gói sẵn `.zip` |

> ⚠️ Cách 2 dùng `gdown --folder`, một số phiên bản **chỉ tải 50 file mỗi thư mục**.
> `run.load_data` đối chiếu với `README.txt` và báo `[THIẾU]` nếu không đủ 240/60 ảnh —
> gặp cảnh báo đó thì chuyển sang cách 1.

In [ ]:
from pipeline import Config, run, report

# --- Cách 1 (khuyến nghị): gắn Drive, đọc thẳng, không tải ------------
# Thư mục phải nằm trong "Drive của tôi". Nếu nó đang ở "Được chia sẻ với tôi",
# bấm chuột phải vào VAI_NVS_DATA_ROUND2 -> "Sắp xếp" -> "Thêm lối tắt vào Drive".
cfg = Config(
    drive_mount=True,
    scene_root="/content/drive/MyDrive/VAI_NVS_DATA_ROUND2",
    drive_subdir="HCM0539",        # chỉ dùng scene này; None = lấy hết
    dataset_url=None,

    data_root="/content/data",
    scenes=(),                     # () = mọi scene tìm thấy dưới scene_root
    resolution=2,                  # ảnh train giảm 2x cho vừa T4
    iterations=7000,
    score_every=1000,              # tần suất chấm điểm khi train
    eval_views=6,
    psnr_max=30.0,                 # PSNR_max của ban tổ chức
    submission_resolution=1,       # render đúng kích thước ảnh gốc

    # --- chống mất model khi phiên Colab bị ngắt ---------------------
    autosave_to_drive=True,        # chép .ply sang Drive NGAY sau mỗi scene
    drive_run_dir="fastgs_runs",   # -> /content/drive/MyDrive/fastgs_runs/<scene>/
    save_to_drive=True,            # submission.zip + models.zip cũng lên Drive
    run_smoke=True,                # False = Run All bỏ qua bước chạy thử

    # --- submission ---------------------------------------------------
    submission_order="csv",        # thứ tự 0001.png theo dòng trong test_poses.csv
)

# --- Cách 2: tải bằng gdown (thay 4 dòng đầu của Config ở trên) -------
#     drive_mount=False,
#     scene_root=None,
#     drive_folder_url="https://drive.google.com/drive/folders/1QqEkZevaEaB7NYZwfE--KlDnn21B3qFz",
#     drive_subdir=None,           # link trên đã trỏ thẳng HCM0539 rồi
#
# Link thư mục cha (mọi scene) — dùng kèm drive_subdir="HCM0539":
#   https://drive.google.com/drive/folders/1zeov1zNpcmgTit9IuJnH38OEAFt79BXI

cfg.show()

## 3 — Load the dataset and profile it

`load_data` lần lượt: lấy dữ liệu → thu hẹp vào `drive_subdir` → tìm scene →
đối chiếu số ảnh với `README.txt` → in hồ sơ.

Scene được đặt tên theo thư mục cha (`HCM0539`), không phải `train`, nên
`submission.zip` sẽ có đúng cấu trúc `HCM0539/0001.png`.

In [ ]:
scenes, profile = run.load_data(cfg)
display(profile)

## 4 — Quick test (one scene, a few hundred iterations)

Proves data + CUDA + scoring all work before spending hours on the real run.

In [ ]:
run.smoke_test(cfg, scenes[0])

## 5 — Train

Thanh tiến trình hiển thị **phần trăm**, loss, số Gaussian và **Score** mới nhất. Cứ
`score_every` vòng lại in một dòng đầy đủ: Score và mức thay đổi, PSNR (kèm `psnr_norm`),
SSIM, LPIPS, RAM và VRAM.

Sau **mỗi** scene, theo đúng thứ tự:

1. `.ply` được chép lên Drive ngay (`autosave_to_drive`) — mất phiên Colab vẫn còn model
2. giải phóng RAM/VRAM
3. render test pose (**chỉ nạp test camera**, không nạp 240 ảnh train)
4. giải phóng lần nữa, rồi ghi `results.json`

Nhờ vậy bộ nhớ không tích luỹ giữa các scene, và không có bước nào chỉ tồn tại ở cuối
đường chạy.

In [ ]:
results, submissions = run.run_all(cfg, scenes)

## 6 — Data analytics: compare the scenes

In [ ]:
history, board = run.analytics(cfg, results, submissions)
display(board)

In [ ]:
report.show_samples(cfg, scenes[0], n=3)      # renders next to ground truth

## 7 — Build `submission.zip` and download it

```
submission.zip
├── <scene>/0001.png, 0002.png, ...
└── ...
```

Test camera lấy từ **`test/test_poses.csv`** của ban tổ chức: đúng 60 pose, render đúng
`width`×`height` ghi trong file, đánh số theo thứ tự dòng (`submission_order="csv"`).
Scene nào không có CSV thì quay về cách cũ (`llffhold`).

`run.finish` nén ảnh, rồi **đối chiếu thẳng với CSV**: đủ số pose chưa, kích thước ảnh có
khớp không, tên file có liên tục không. Với `save_to_drive=True` mọi thứ đã nằm trên Drive
trước khi tải, nên trình duyệt chặn tải cũng không mất gì.

In [ ]:
check, problems = run.finish(cfg, scenes)
display(check)